In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
import numpy as np

# a. Set up environment & b. Loading and preprocessing data
IMG_SIZE = 224
BATCH_SIZE = 32

# Load dataset directly from the main folder, splitting 80% for training and 20% for validation
train_ds = tf.keras.utils.image_dataset_from_directory(
    'caltech-101',
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    'caltech-101',
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# Keep a separate un-preprocessed validation dataset just for clean visualization
viz_ds = test_ds

# Preprocessing function specific to ResNet for training/evaluation
def preprocess(images, labels):
    return tf.keras.applications.resnet50.preprocess_input(images), labels

train_ds = train_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

# c. Choose the pre-trained Model (Transfer Learning)
base_model = tf.keras.applications.ResNet50(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# d. Freeze the base layers and add custom layers for fine tuning
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# e. Object detection / classification training & f. Compute loss
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("--- Training Initial Custom Head ---")
history = model.fit(train_ds, epochs=5, validation_data=test_ds)

# g. Evaluate the Model
loss, accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# h. Visualize the prediction (using raw images from viz_ds and preprocessed for prediction)
for raw_images, labels in viz_ds.take(1):
    # Preprocess a copy for the model prediction
    processed_input = tf.keras.applications.resnet50.preprocess_input(raw_images)
    preds = model.predict(processed_input)
    
    pred_idx = np.argmax(preds[0])
    true_idx = np.argmax(labels[0].numpy())
    
    plt.figure(figsize=(6, 6))
    plt.imshow(raw_images[0].numpy().astype("uint8"))
    plt.title(f"True: {class_names[true_idx]} | Pred: {class_names[pred_idx]}")
    plt.axis('off')
    plt.show()
    break

# i. Fine Tune the Model (Unfreeze top layers of base model)
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5), # Lower learning rate for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("--- Fine-Tuning Model ---")
history_fine = model.fit(train_ds, epochs=5, validation_data=test_ds)